El paquete de _Python_ [scikit-learn](http://scikit-learn.org) (_sklearn_ en lo que sigue) proporciona un marco de trabajo para el aprendizaje automático.

# Aprendizaje supervisado

Para ilustrar el concepto de aprendizaje supervisado vamos a usar el conjunto de datos [_Car Evaluation_](http://archive.ics.uci.edu/ml/datasets/Car+Evaluation) del repositorio [UCI](http://archive.ics.uci.edu/ml/). Este conjunto de datos contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos:
* _WS1_: intensidad de la señal del router 1 en dBm. Valores entre `-74` y `-10`.
* _WS2_: intensidad de la señal del router 2 en dBm. Valores entre `-74` y `-45`.
* _WS3_: intensidad de la señal del router 3 en dBm. Valores entre `-73` y `-40`.
* _WS4_: intensidad de la señal del router 4 en dBm. Valores entre `-77` y `-11`.
* _WS5_: intensidad de la señal del router 5 en dBm. Valores entre `-89` y `-36`.
* _WS6_: intensidad de la señal del router 6 en dBm. Valores entre `-97` y `-61`.
* _WS7_: intensidad de la señal del router 7 en dBm. Valores entre `-98` y `-63`

La habitación en la que se encontraba el dispositivo se indica mediante el atributo _Room_, que puede tomar los valores `1`, `2`, `3` o `4`.

Para leer los datos desde el fichero `cars.csv` que se proporciona se pueden evaluar las siguientes expresiones ([_Pandas_](http://pandas.pydata.org/) y [_NumPy_](http://www.numpy.org/) son paquetes de _Python_ para análisis de datos y cálculo científico, respectivamente):

In [1]:
# Instalar paquetes pandas y numpy, pip install pandas numpy
import pandas
import numpy

wifi = pandas.read_csv('wifi_localization_excel.csv', header=None, sep='\t',
                       names=['WS1', 'WS2', 'WS3', 'WS4', 'WS5', 'WS6', 'WS7', 'Room'])

FileNotFoundError: [Errno 2] No such file or directory: 'wifi_localization_excel.csv'

In [1]:
# Número de filas y columnas
print(wifi.shape)

NameError: name 'cars' is not defined

In [4]:
# 10 primeras filas
wifi.head(10)

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc
5,vhigh,vhigh,2,2,med,high,unacc
6,vhigh,vhigh,2,2,big,low,unacc
7,vhigh,vhigh,2,2,big,med,unacc
8,vhigh,vhigh,2,2,big,high,unacc
9,vhigh,vhigh,2,4,small,low,unacc


_sklearn_ no puede trabajar directamente con el conjunto de datos anterior, ya que asume que los valores de las variables discretas están codificadas con números enteros. Para transformar los datos a un formato adecuado ofrece diversas operaciones de preprocesamiento, entre las que se encuentran `OrdinalEncoder`, para codificar los atributos, y `LabelEncoder`, para codificar la variable objetivo.

In [3]:
# Instalar paquete scikit-learn, pip install scikit-learn
from sklearn import preprocessing

atributos = cars.loc[:, 'buying':'safety']  # selección de las columnas de atributos
objetivo = cars['acceptability']  # selección de la columna objetivo

# Para realizar una codificación de los datos, se crea una instancia del tipo de
# codificación pretendida y se ajusta a los datos disponibles mediante el método
# fit. Los métodos transform e inverse_transform permiten entonces codificar y
# descodificar, respectivamente, los datos.

In [4]:
# El codificador adecuado para los atributos es OrdinalEncoder, ya que permite
# trabajar con el array completo de valores de los atributos.
codificador_atributos = preprocessing.OrdinalEncoder()
codificador_atributos.fit(atributos)

# Categorías detectadas por el codificador para cada atributo
print(codificador_atributos.categories_)

[array(['high', 'low', 'med', 'vhigh'], dtype=object), array(['high', 'low', 'med', 'vhigh'], dtype=object), array(['2', '3', '4', '5more'], dtype=object), array(['2', '4', 'more'], dtype=object), array(['big', 'med', 'small'], dtype=object), array(['high', 'low', 'med'], dtype=object)]


In [5]:
# Codificación de los atributos una vez ajustado el codificador
atributos_codificados = codificador_atributos.transform(atributos)
print(atributos_codificados)

[[3. 3. 0. 0. 2. 1.]
 [3. 3. 0. 0. 2. 2.]
 [3. 3. 0. 0. 2. 0.]
 ...
 [1. 1. 3. 2. 0. 1.]
 [1. 1. 3. 2. 0. 2.]
 [1. 1. 3. 2. 0. 0.]]


In [6]:
# Las respuestas predichas por los modelos estarán codificadas, pero el codificador
# también permite descodificarlas
print(codificador_atributos.inverse_transform([[3., 3., 0., 0., 2., 1.],
                                               [1., 1., 3., 2., 0., 1.]]))

[['vhigh' 'vhigh' '2' '2' 'small' 'low']
 ['low' 'low' '5more' 'more' 'big' 'low']]


In [7]:
# El codificador adecuado para la variable objetivo es LabelEncoder, que trabaja
# con la lista de sus valores, en lugar de con un array.
codificador_objetivo = preprocessing.LabelEncoder()

# El método fit_transform ajusta la codificación y la aplica a los datos a continuación
objetivo_codificado = codificador_objetivo.fit_transform(objetivo)

In [8]:
# Clases detectadas por el codificador para la variable objetivo
print(codificador_objetivo.classes_)

['acc' 'good' 'unacc' 'vgood']


In [10]:
print(objetivo_codificado)

[2 2 2 ... 2 1 3]


In [11]:
print(codificador_objetivo.inverse_transform([2, 1, 3]))

['unacc' 'good' 'vgood']


Una vez codificadas las variables, es necesario separar el conjunto de datos en dos: un conjunto de entrenamiento, que se usará para construir los distintos modelos; y un conjunto de prueba, que se usará para comparar los distintos modelos.

Un detalle a tener en cuenta es que la distribución de ejemplos en las distintas clases de aceptabilidad no es uniforme: hay 1210 coches (un 70.023 % del total) clasificados como inaceptables (`unacc`), 384 coches (22.222 %) clasificados como aceptables (`acc`), 69 coches (3.993 %) clasificados como buenos (`good`) y 65 coches (3.762 %) clasificados como muy buenos (`vgood`).

In [12]:
# Cantidad total de ejemplos
print(cars.shape[0])

1728


In [13]:
# Frecuencia total de cada clase de aceptabilidad
print(pandas.Series(objetivo).value_counts(normalize=True))

unacc    0.700231
acc      0.222222
good     0.039931
vgood    0.037616
Name: acceptability, dtype: float64


Es conveniente, por tanto, que la separación de los ejemplos se realice de manera estratificada, es decir, intentando mantener la proporción anterior tanto en el conjunto de entrenamiento como en el de prueba.

Para dividir un conjunto de datos en un subconjunto de entrenamiento y otro de prueba, _sklearn_ proporciona la función `train_test_split`.

In [9]:
from sklearn import model_selection

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = model_selection.train_test_split(
        # Conjuntos de datos a dividir, usando los mismos índices para ambos
        atributos_codificados, objetivo_codificado,
        # Valor de la semilla aleatoria, para que el muestreo sea reproducible,
        # a pesar de ser aleatorio
        random_state=12345,
        # Tamaño del conjunto de prueba
        test_size=.33,
        # Estratificamos respecto a la distribución de valores en la variable objetivo
        stratify=objetivo_codificado)

In [10]:
# Comprobamos que el conjunto de prueba contiene el 33 % de los datos, en la misma proporción
# con respecto a la variable objetivo
print("Cantidad de ejemplos de pruebas requeridos:", 1728 * .33)
print("Filas del array de atributos de prueba:", atributos_prueba.shape[0])
print("Longitud del vector de objetivos de prueba:", len(objetivo_prueba))
print("Proporción de clases en el vector de objetivos de prueba:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_prueba)
      ).value_counts(normalize=True))

Cantidad de ejemplos de pruebas requeridos: 570.24
Filas del array de atributos de prueba: 571
Longitud del vector de objetivos de prueba: 571
Proporción de clases en el vector de objetivos de prueba:
unacc    0.700525
acc      0.222417
good     0.040280
vgood    0.036778
Name: proportion, dtype: float64


In [11]:
# Comprobamos que el conjunto de entrenamiento contiene el resto de los datos, en la misma
# proporción con respecto a la variable objetivo
print("Cantidad de ejemplos de entrenamiento requeridos:", 1728 * .67)
print("Filas del array de atributos de entrenamiento:", atributos_entrenamiento.shape[0])
print("Longitud del vector de objetivos de entrenamiento:", len(objetivo_entrenamiento))
print("Proporción de clases en el vector de objetivos de entrenamiento:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_entrenamiento)
      ).value_counts(normalize=True))

Cantidad de ejemplos de entrenamiento requeridos: 1157.76
Filas del array de atributos de entrenamiento: 1157
Longitud del vector de objetivos de entrenamiento: 1157
Proporción de clases en el vector de objetivos de entrenamiento:
unacc    0.700086
acc      0.222126
good     0.039758
vgood    0.038029
Name: proportion, dtype: float64


Para realizar aprendizaje supervisado en _sklearn_ basta crear una instancia de la clase de objetos que implemente el modelo que se quiera utilizar (árboles de decisión, _naive_ Bayes, _kNN_, etc.).

Cada una de estas instancias dispondrá de los siguientes métodos:
* El método `fit` permite entrenar el modelo, dados __por separado__ el conjunto de ejemplos de entrenamiento y la clase de cada uno de estos ejemplos.
* El método `predict` permite clasificar un nuevo ejemplo una vez entrenado el modelo.
* El método `score` calcula el rendimiento del modelo, dados __por separado__ el conjunto de ejemplos de prueba y la clase de cada uno de estos ejemplos.

### Árboles de decisión

_sklearn_ implementa los árboles de decisión clasificadores como instancias de la clase `DecisionTreeClassifier`.

Desafortunadamente, son árboles de decisión binarios construidos asumiendo atributos continuos y mediante un algoritmo distinto a _ID3_, que no está implementado.

En http://scikit-learn.org/stable/modules/tree.html se puede encontrar información acerca de los árboles de decisión implementados en _sklearn_.

In [13]:
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# En scikit-learn, ID3 se especifica mediante criterion='entropy'.
# Nota: La implementación nativa de sklearn soporta divisiones numéricas/ordinales.
modelo_id3 = DecisionTreeClassifier(
    criterion='entropy',  # Usa Ganancia de Información (Entropía) como en ID3
    random_state=12345
)

# Entrenamiento del modelo
modelo_id3.fit(atributos_entrenamiento, objetivo_entrenamiento)

# 7. Evaluación del modelo
predicciones = modelo_id3.predict(atributos_prueba)

# Métricas de precisión general
precision_general = accuracy_score(objetivo_prueba, predicciones)
print(f"Exactitud (Accuracy) del modelo ID3: {precision_general * 100:.2f}%\n")

# Reporte detallado por clase (Precision, Recall, F1-Score)
etiquetas_clases = codificador_objetivo.classes_
print("=== Reporte de Clasificación ===")
print(classification_report(
    objetivo_prueba, 
    predicciones, 
    target_names=etiquetas_clases
))

# Matriz de confusión
print("=== Matriz de Confusión ===")
matriz_confusion = pandas.DataFrame(
    confusion_matrix(objetivo_prueba, predicciones),
    index=[f"Real: {clase}" for clase in etiquetas_clases],
    columns=[f"Pred: {clase}" for clase in etiquetas_clases]
)
print(matriz_confusion)

# 8. Visualización en texto de la estructura del árbol
print("\n=== Reglas del Árbol de Decisión ===")
reglas_arbol = export_text(modelo_id3, feature_names=list(atributos.columns))
print(reglas_arbol[:1000])  # Muestra las primeras reglas generadas

Exactitud (Accuracy) del modelo ID3: 96.67%

=== Reporte de Clasificación ===
              precision    recall  f1-score   support

         acc       0.94      0.91      0.93       127
        good       0.95      0.91      0.93        23
       unacc       0.97      0.99      0.98       400
       vgood       1.00      0.95      0.98        21

    accuracy                           0.97       571
   macro avg       0.97      0.94      0.95       571
weighted avg       0.97      0.97      0.97       571

=== Matriz de Confusión ===
             Pred: acc  Pred: good  Pred: unacc  Pred: vgood
Real: acc          116           1           10            0
Real: good           1          21            1            0
Real: unacc          5           0          395            0
Real: vgood          1           0            0           20

=== Reglas del Árbol de Decisión ===
|--- persons <= 0.50
|   |--- class: 2
|--- persons >  0.50
|   |--- safety <= 0.50
|   |   |--- maint <= 2.50
|   |

In [19]:
# =============================================================================
# 9. Predicción con datos nuevos
# =============================================================================

# Definimos una o varias instancias de coches nuevos con sus valores originales
nuevos_coches = pandas.DataFrame([
    {
        'buying': 'vhigh',
        'maint': 'high',
        'doors': '2',
        'persons': '2',
        'lug_boot': 'small',
        'safety': 'low'
    },
    {
        'buying': 'low',
        'maint': 'low',
        'doors': '4',
        'persons': '4',
        'lug_boot': 'big',
        'safety': 'high'
    }
])

print("--- Datos nuevos a evaluar ---")
print(nuevos_coches)

# Usamos .transform() 
nuevos_atributos_codificados = codificador_atributos.transform(nuevos_coches)

# Realizamos la predicción numérica con el modelo ID3
predicciones_numericas = modelo_id3.predict(nuevos_atributos_codificados)

# Convertimos la predicción numérica de vuelta a su etiqueta original ('unacc', 'acc', etc.)
predicciones_etiquetas = codificador_objetivo.inverse_transform(predicciones_numericas)

# Mostramos los resultados en un DataFrame claro
resultados = nuevos_coches.copy()
resultados['prediccion_aceptabilidad'] = predicciones_etiquetas

print("\n--- Resultados de la predicción ---")
print(resultados['prediccion_aceptabilidad'])

--- Datos nuevos a evaluar ---
  buying maint doors persons lug_boot safety
0  vhigh  high     2       2    small    low
1    low   low     4       4      big   high

--- Resultados de la predicción ---
0    unacc
1    vgood
Name: prediccion_aceptabilidad, dtype: str


_sklearn_ implementa _naive_ Bayes para atributos categóricos mediante instancias de la clase `CategoricalNB`.

In [17]:
from sklearn import naive_bayes

clasif_NB = naive_bayes.CategoricalNB(alpha=1.0)  # alpha es el parámetro de suavizado
clasif_NB.fit(atributos_entrenamiento, objetivo_entrenamiento)

CategoricalNB()

Las siguientes expresiones muestran las cuentas realizadas y (los logaritmos de) las probabilidades aprendidas por el modelo.

In [18]:
for clase, cantidad_ejemplos_clase, log_probabilidad_clase in zip(
    codificador_objetivo.classes_, clasif_NB.class_count_, clasif_NB.class_log_prior_):
        print("Cantidad de ejemplos para la clase", clase, ":", cantidad_ejemplos_clase)
        print("Logaritmo de la probabilidad aprendida para la clase", clase, ":", log_probabilidad_clase)

Cantidad de ejemplos para la clase acc : 257.0
Logaritmo de la probabilidad aprendida para la clase acc : -1.5045096422984567
Cantidad de ejemplos para la clase good : 46.0
Logaritmo de la probabilidad aprendida para la clase good : -3.2249443307045818
Cantidad de ejemplos para la clase unacc : 810.0
Logaritmo de la probabilidad aprendida para la clase unacc : -0.3565514795271927
Cantidad de ejemplos para la clase vgood : 44.0
Logaritmo de la probabilidad aprendida para la clase vgood : -3.269396093275416


In [19]:
for i, atributo in enumerate(cars.columns[:-1]):
    print("Cantidad de ejemplos para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.category_count_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print("Logaritmos de las probabilidades aprendidas para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.feature_log_prob_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print()

Cantidad de ejemplos para el atributo buying :
        high    low    med  vhigh
acc     69.0   64.0   70.0   54.0
good     0.0   29.0   17.0    0.0
unacc  230.0  177.0  174.0  229.0
vgood    0.0   26.0   18.0    0.0
Logaritmos de las probabilidades aprendidas para el atributo buying :
           high       low       med     vhigh
acc   -1.316025 -1.390133 -1.301841 -1.557187
good  -3.912023 -0.510826 -1.021651 -3.912023
unacc -1.259543 -1.520177 -1.537174 -1.263881
vgood -3.871201 -0.575364 -0.926762 -3.871201

Cantidad de ejemplos para el atributo maint :
        high    low    med  vhigh
acc     62.0   65.0   78.0   52.0
good     0.0   33.0   13.0    0.0
unacc  205.0  186.0  174.0  245.0
vgood   12.0   17.0   15.0    0.0
Logaritmos de las probabilidades aprendidas para el atributo maint :
           high       low       med     vhigh
acc   -1.421386 -1.374866 -1.195073 -1.594228
good  -3.912023 -0.385662 -1.272966 -3.912023
unacc -1.374084 -1.470852 -1.537174 -1.196629
vgood -1.3062

El método `predict` devuelve la clase predicha por el modelo para un nuevo ejemplo y el método `score` la exactitud (_accuracy_) media sobre un conjunto de datos de prueba.

In [20]:
nuevos_ejemplos = [['vhigh', 'vhigh', '3', 'more', 'big', 'high'],
                   ['high', 'low', '3', '2', 'med', 'med']]
codificador_objetivo.inverse_transform(clasif_NB.predict(codificador_atributos.transform(nuevos_ejemplos)))

array(['unacc', 'unacc'], dtype=object)

In [21]:
# Calculamos la fracción de clases correctamente predichas para el conjunto de datos de prueba
clasif_NB.score(atributos_prueba, objetivo_prueba)

0.8441330998248686